# Polar MHW Heat Budget: Onset vs Decline Drivers

Analyses a single 10×10 grid-cell box (2.5°×2.5° at 0.25° resolution) in the Antarctic
Peninsula / Drake Passage region.  For every MHW detected in the full IAF cycle-6 run
(1958–2019), we:
1. Detect events using a Hobday et al. (2016) threshold exceedance.
2. Separate onset (start→peak) from decline (peak→end) phases.
3. Compute the mean mixed-layer heat budget anomaly for each phase.
4. Composite across all events to identify dominant drivers.

Closely follows the approach in `MHW_example_analysis.ipynb` (Holmes & Malan, in prep.).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import NullFormatter
from matplotlib.patches import Rectangle
import pandas as pd
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature

%matplotlib inline

In [ ]:
from dask.distributed import Client
client = Client()
client

## 1. Configuration

In [ ]:
def map_lon_to_ds(lon_e, lon_coord_min=-280.0, lon_coord_max=80.0):
    """Map standard longitudes to the ACCESS-OM2 grid convention."""
    lonmin, lonmax = lon_coord_min, lon_coord_max
    if 0.0 <= lonmin and lonmax <= 360.0:
        return lon_e % 360.0
    if -180.0 <= lonmin and lonmax <= 180.0:
        return ((lon_e + 180.0) % 360.0) - 180.0
    if lonmax <= 100.0 and lonmin < -180.0:  # ACCESS-style (-280 to 80)
        return lon_e if lon_e <= lonmax else lon_e - 360.0
    return lon_e

# ── Study box: 10×10 grid cells (2.5°×2.5°) in the Drake Passage / Antarctic Peninsula margin
# Centred near 71.25°W, 65.25°S.  Adjust sreg to move the box.
# Format: [lon_min, lon_max, lat_min, lat_max]  (ACCESS-OM2 grid convention)
sreg = [map_lon_to_ds(287.5), map_lon_to_ds(290.0), -66.5, -64.0]  # 72.5°W–70°W, 66.5°S–64°S

# Wider region for context maps
reg  = [map_lon_to_ds(250.0), map_lon_to_ds(320.0), -80.0, -55.0]

# ── Data path (ACCESS-OM2 0.25° IAF cycle 6 on Gadi)
base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'

# Cycle 6 output numbering: output305 = 1958, output366 = 2019
# Derived from reference notebook: output358 = 2011, output364 = 2017
# Starting with last 10 years (2010–2019) until full budget file availability is confirmed.
# Verify available outputs: ls /g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/ | grep output
START_OUTPUT = 357   # 2010
END_OUTPUT   = 366   # 2019
outputs = list(range(START_OUTPUT, END_OUTPUT + 1))

# ── MHW detection parameters (Hobday et al. 2016)
MIN_DUR = 5   # minimum event duration (days)
MAX_GAP = 2   # maximum gap (days) to bridge between exceedances

# ── Dask chunk sizes
chunks2D = {'time': 1, 'yt_ocean': 216, 'xt_ocean': 240}

SEC_PER_DAY = 86400.0

# Budget terms to analyse (derived: Surface Flux = surface_flux + sw_pen)
BUDGET_TERMS = ['MLT tendency', 'Surface Flux', 'Advection', 'Vertical mixing', 'Entrainment']

## 2. Context map

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax  = plt.axes(projection=ccrs.SouthPolarStereo())
ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='0.85', edgecolor='k', linewidth=0.5, zorder=2)
ax.coastlines(linewidth=0.7, zorder=3)
ax.gridlines(draw_labels=False, linestyle=':', linewidth=0.4, alpha=0.5)

# Study box in PlateCarree then transformed
rect = Rectangle(
    (sreg[0], sreg[2]), sreg[1] - sreg[0], sreg[3] - sreg[2],
    facecolor='tab:red', edgecolor='darkred', linewidth=1.5,
    alpha=0.6, transform=ccrs.PlateCarree(), zorder=4,
)
ax.add_patch(rect)
ax.set_title(
    f'Study box: {abs(sreg[2]):.1f}°S–{abs(sreg[3]):.1f}°S, '
    f'{abs(sreg[0]):.1f}°W–{abs(sreg[1]):.1f}°W',
    fontsize=12
)
plt.tight_layout()

## 3. Load mixed layer temperature — full time series

In [ ]:
sst_files = [base + f'output{o:03d}/ocean/ocean_daily.nc' for o in outputs]

ds_sst = xr.open_mfdataset(
    sst_files,
    decode_times=True,
    chunks=chunks2D,
    combine='nested',
    concat_dim='time',
    data_vars=['temp_in_mld', 'mld'],
    parallel=True,
    decode_timedelta=False,
).sel(
    xt_ocean=slice(sreg[0], sreg[1]),
    yt_ocean=slice(sreg[2], sreg[3]),
)

temp = ds_sst['temp_in_mld'] / 1035   # unit conversion matching reference notebook

# Area-weighted spatial mean over box
w = np.cos(np.deg2rad(ds_sst.yt_ocean))
temp_mean = temp.weighted(w).mean(('xt_ocean', 'yt_ocean'))
mld_mean  = ds_sst.mld.weighted(w).mean(('xt_ocean', 'yt_ocean'))

print("SST files opened (lazy). Time range:")
print(f"  {len(outputs)} years | {len(sst_files)} files")

## 4. Load climatology and threshold (pre-computed)

In [ ]:
clim   = xr.open_dataset(base + 'post_processed_diags/om2_025_MLT_clim.nc').sel(
    xt_ocean=slice(sreg[0], sreg[1]), yt_ocean=slice(sreg[2], sreg[3]))
thresh = xr.open_dataset(base + 'post_processed_diags/om2_025_MLT_thresh.nc').sel(
    xt_ocean=slice(sreg[0], sreg[1]), yt_ocean=slice(sreg[2], sreg[3]))

w = np.cos(np.deg2rad(clim.yt_ocean))
clim_mean   = clim.temp.weighted(w).mean(('xt_ocean', 'yt_ocean')).compute()
thresh_mean = thresh.temp.weighted(w).mean(('xt_ocean', 'yt_ocean')).compute()

# Map DOY climatology onto actual daily time axis of the full run
print("Loading temperature time series (may take a few minutes) ...")
temp_vals = temp_mean.compute()
print("Done.")

doy_all        = temp_vals.time.dt.dayofyear
clim_on_time   = clim_mean.sel(dayofyear=doy_all).assign_coords(time=temp_vals.time)
thresh_on_time = thresh_mean.sel(dayofyear=doy_all).assign_coords(time=temp_vals.time)

## 5. MHW detection — all events in the full time series

In [ ]:
def detect_mhw_events(temp_da, thresh_da, min_dur=5, max_gap=2):
    """
    Hobday et al. (2016) event detection.
    Returns list of dicts with keys: t_start, t_peak, t_end.
    t_peak is the day of maximum anomaly above threshold.
    """
    t      = pd.to_datetime(temp_da.time.values)
    y_temp = temp_da.values
    y_thr  = thresh_da.values
    valid  = np.isfinite(y_temp) & np.isfinite(y_thr)

    exceed = pd.Series((y_temp[valid] > y_thr[valid]).astype(int), index=t[valid])

    # Bridge short gaps (<= max_gap days) between exceedances
    runs        = (exceed.diff(1).ne(0)).cumsum()
    run_lengths = exceed.groupby(runs).transform('size')
    short_false = (exceed == 0) & (run_lengths <= max_gap)
    merged      = exceed.copy()
    merged[short_false] = 1

    # Keep only runs lasting >= min_dur days
    runs2      = (merged.diff(1).ne(0)).cumsum()
    lens2      = merged.groupby(runs2).transform('size')
    long_true  = (merged == 1) & (lens2 >= min_dur)

    # Extract contiguous event blocks
    anom = pd.Series(y_temp[valid] - y_thr[valid], index=t[valid])
    arr, idx = long_true.to_numpy(), long_true.index.to_numpy()
    events, start_i = [], None
    for i in range(len(arr)):
        if arr[i] and start_i is None:
            start_i = i
        if start_i is not None and (i == len(arr) - 1 or not arr[i + 1]):
            seg    = anom.iloc[start_i:i + 1]
            t_peak = seg.idxmax()
            events.append({'t_start': idx[start_i], 't_peak': t_peak, 't_end': idx[i]})
            start_i = None
    return events


events = detect_mhw_events(temp_vals, thresh_on_time, min_dur=MIN_DUR, max_gap=MAX_GAP)
print(f"Detected {len(events)} MHW events\n")

ev_df = pd.DataFrame(events)
ev_df['onset_days']   = (pd.to_datetime(ev_df.t_peak) - pd.to_datetime(ev_df.t_start)).dt.days
ev_df['decline_days'] = (pd.to_datetime(ev_df.t_end)  - pd.to_datetime(ev_df.t_peak)).dt.days
ev_df['duration']     = ev_df['onset_days'] + ev_df['decline_days']
print(ev_df[['t_start', 't_peak', 't_end', 'onset_days', 'decline_days', 'duration']].to_string())

### Overview: all events on the temperature time series

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

temp_vals.plot(ax=ax, color='steelblue', linewidth=0.6, label='MLT')
thresh_on_time.plot(ax=ax, color='tab:red', linewidth=0.9, linestyle='--', alpha=0.7, label='Threshold (90th %)')
clim_on_time.plot(ax=ax, color='k', linewidth=0.7, linestyle=':', alpha=0.5, label='Climatology')

for k, ev in enumerate(events):
    ax.axvspan(ev['t_start'], ev['t_end'], alpha=0.2, color='tab:orange',
               label='MHW' if k == 0 else '_')

ax.set_title(f'All detected MHW events — {len(events)} total, 2010–2019', fontsize=12)
ax.set_xlabel('Time')
ax.set_ylabel('MLT (°C)')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()

## 6. Load budget data — full time series

In [ ]:
budget_files = [
    base + f'post_processed_diags/mlt_budget_online_stavg/mlt_budget_stavg_daily_online_output{o:03d}.nc'
    for o in outputs
]

vars_raw = ['mlt_tendency', 'advection', 'vert_mixing', 'entrainment', 'surface_flux', 'sw_pen', 'residual']

budget_raw = xr.open_mfdataset(
    budget_files,
    decode_times=True,
    chunks=chunks2D,
    combine='nested',
    concat_dim='time',
    parallel=True,
    decode_timedelta=False,
).sel(
    xt_ocean=slice(sreg[0], sreg[1]),
    yt_ocean=slice(sreg[2], sreg[3]),
)[vars_raw]

# Derived: net surface heat flux absorbed in mixed layer
budget_raw['surf_to_ML'] = budget_raw['surface_flux'] + budget_raw['sw_pen']

# Area-weighted mean over box, convert degC s-1 → degC day-1
w = np.cos(np.deg2rad(budget_raw.yt_ocean))
budget_series = xr.Dataset({
    'MLT tendency'   : (budget_raw['mlt_tendency'].weighted(w).mean(('yt_ocean', 'xt_ocean')) * SEC_PER_DAY),
    'Surface Flux'   : (budget_raw['surf_to_ML'].weighted(w).mean(('yt_ocean', 'xt_ocean'))   * SEC_PER_DAY),
    'Advection'      : (budget_raw['advection'].weighted(w).mean(('yt_ocean', 'xt_ocean'))     * SEC_PER_DAY),
    'Vertical mixing': (budget_raw['vert_mixing'].weighted(w).mean(('yt_ocean', 'xt_ocean'))   * SEC_PER_DAY),
    'Entrainment'    : (budget_raw['entrainment'].weighted(w).mean(('yt_ocean', 'xt_ocean'))   * SEC_PER_DAY),
    'Residual'       : (budget_raw['residual'].weighted(w).mean(('yt_ocean', 'xt_ocean'))      * SEC_PER_DAY),
}).chunk({'time': -1})

print("Budget files opened (lazy).")

## 7. Budget climatology

In [ ]:
budget_clim_ds = xr.open_dataset(
    base + 'post_processed_diags/mlt_budget_online_stavg/'
           'mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc'
).sel(xt_ocean=slice(sreg[0], sreg[1]), yt_ocean=slice(sreg[2], sreg[3]))

budget_clim_ds['surf_to_ML'] = budget_clim_ds['surface_flux'] + budget_clim_ds['sw_pen']

w = np.cos(np.deg2rad(budget_clim_ds.yt_ocean))
budget_clim = xr.Dataset({
    'MLT tendency'   : (budget_clim_ds['mlt_tendency'].weighted(w).mean(('yt_ocean', 'xt_ocean')) * SEC_PER_DAY),
    'Surface Flux'   : (budget_clim_ds['surf_to_ML'].weighted(w).mean(('yt_ocean', 'xt_ocean'))   * SEC_PER_DAY),
    'Advection'      : (budget_clim_ds['advection'].weighted(w).mean(('yt_ocean', 'xt_ocean'))     * SEC_PER_DAY),
    'Vertical mixing': (budget_clim_ds['vert_mixing'].weighted(w).mean(('yt_ocean', 'xt_ocean'))   * SEC_PER_DAY),
    'Entrainment'    : (budget_clim_ds['entrainment'].weighted(w).mean(('yt_ocean', 'xt_ocean'))   * SEC_PER_DAY),
}).compute()

print("Budget climatology loaded.")
print(budget_clim)

## 8. Full-series budget anomalies

Monthly climatology → interpolated to daily → smoothed with a 31-day window
(consistent with MHW threshold calculation).

In [ ]:
# Collapse to 12 calendar months, pad for periodicity, interpolate to daily
clim_by_month = (
    budget_clim
    .groupby('time.month').mean('time')
    .assign_coords(month=np.arange(1, 13))
)
dec = clim_by_month.sel(month=12).expand_dims(month=[0])
jan = clim_by_month.sel(month=1).expand_dims(month=[13])
clim_padded = xr.concat([dec, clim_by_month, jan], dim='month')

days_interp  = np.linspace(1, 12, 365)
clim_smooth  = clim_padded.interp(month=days_interp)

doys_budget  = (budget_series.time.dt.dayofyear - 1) % 365
clim_daily   = xr.Dataset({
    name: clim_smooth[name].isel(month=doys_budget).assign_coords(time=budget_series.time)
    for name in BUDGET_TERMS
})
clim_daily = clim_daily.rolling(time=31, center=True, min_periods=1).mean()

# Anomaly relative to smoothed monthly climatology
anom_series = xr.Dataset(
    {name: budget_series[name] - clim_daily[name] for name in BUDGET_TERMS}
)

print("Loading budget anomalies for full time series (may take several minutes) ...")
anom_series = anom_series.compute()
print("Done.")

## 9. Onset vs decline budget decomposition

In [ ]:
rows = []
for ev in events:
    t_s, t_p, t_e = ev['t_start'], ev['t_peak'], ev['t_end']
    for name in BUDGET_TERMS:
        onset_anom   = float(anom_series[name].sel(time=slice(t_s, t_p)).mean())
        decline_anom = float(anom_series[name].sel(time=slice(t_p, t_e)).mean())
        rows.append({
            't_start': t_s,
            't_peak' : t_p,
            't_end'  : t_e,
            'term'   : name,
            'onset'  : onset_anom,
            'decline': decline_anom,
        })

df = pd.DataFrame(rows)

summary = df.groupby('term')[['onset', 'decline']].agg(['mean', 'std']).reindex(BUDGET_TERMS)
print(f"\nComposite over {len(events)} events (°C day⁻¹):")
print(summary.round(4))

## 10. Summary plots

### 10a. Composite bar chart — mean onset vs decline contributions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

mean_onset   = df.groupby('term')['onset'].mean().reindex(BUDGET_TERMS)
mean_decline = df.groupby('term')['decline'].mean().reindex(BUDGET_TERMS)

x     = np.arange(len(BUDGET_TERMS))
width = 0.35

ax.bar(x - width/2, mean_onset,   width, label='Onset',   color='tab:orange', alpha=0.85)
ax.bar(x + width/2, mean_decline, width, label='Decline', color='tab:blue',   alpha=0.85)

# Individual events as scatter overlay
for i, name in enumerate(BUDGET_TERMS):
    ev_o = df[df.term == name]['onset'].values
    ev_d = df[df.term == name]['decline'].values
    ax.scatter(np.full(len(ev_o), i - width/2), ev_o, s=20, c='k', zorder=5, alpha=0.35)
    ax.scatter(np.full(len(ev_d), i + width/2), ev_d, s=20, c='k', zorder=5, alpha=0.35)

ax.axhline(0, color='k', linewidth=0.8, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([t.replace(' ', '\n') for t in BUDGET_TERMS], fontsize=11)
ax.set_ylabel('Budget anomaly (°C day⁻¹)', fontsize=11)
ax.set_title(
    f'Mean budget drivers of MHW onset vs decline\n'
    f'({len(events)} events, 2010–2019  |  '
    f'{abs(sreg[2]):.1f}°S–{abs(sreg[3]):.1f}°S, '
    f'{abs(sreg[0]):.1f}°W–{abs(sreg[1]):.1f}°W)',
    fontsize=12
)
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.5, axis='y')
plt.tight_layout()

### 10b. Box plots — distribution across events

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, phase, color in zip(axes, ['onset', 'decline'], ['tab:orange', 'tab:blue']):
    data = [df[df.term == name][phase].values for name in BUDGET_TERMS]
    bp   = ax.boxplot(data, patch_artist=True,
                      medianprops=dict(color='k', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(0, color='k', linewidth=0.7, alpha=0.5)
    ax.set_xticks(range(1, len(BUDGET_TERMS) + 1))
    ax.set_xticklabels([t.replace(' ', '\n') for t in BUDGET_TERMS], fontsize=10)
    ax.set_title(phase.capitalize(), fontsize=12)
    ax.grid(True, linestyle=':', alpha=0.4, axis='y')

axes[0].set_ylabel('Budget anomaly (°C day⁻¹)', fontsize=11)
fig.suptitle(f'Distribution across {len(events)} events', fontsize=13)
plt.tight_layout()

### 10c. Budget closure check

In [ ]:
rhs_terms = [t for t in BUDGET_TERMS if t != 'MLT tendency']
closure = anom_series['MLT tendency'].copy()
for name in rhs_terms:
    closure = closure - anom_series[name]

fig, ax = plt.subplots(figsize=(14, 3))
closure.plot(ax=ax, color='k', linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.8, alpha=0.5, linestyle='--')
ax.set_title('Budget closure: MLT tendency − ΣRHSterms (should be ≈ 0)')
ax.set_ylabel('°C day⁻¹')
ax.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()